# MatplotlibとSeabornの使い方

可視化は、数字の表をきれいにするための飾りではなく、『いま何が起きているかを一目でつかむ』ための道具です。Matplotlib は土台、Seaborn はその上で統計的な図を手早く描くための補助と考えると分かりやすくなります。

このノートでは、学習の進み具合、2 変数の関係、分布の形、カテゴリごとの差、特徴量同士の関係という 5 つの見方を、図を切り替えながら確かめます。

## まずは『何を見たい図なのか』を決める

可視化で迷いやすいのは、描き方より先に目的が曖昧なままコードを書き始めてしまうことです。増減を見たいのか、関係を見たいのか、ばらつきを見たいのか。そこが決まるだけで、選ぶべき図の候補はかなり絞れます。


このノートでは、1 枚ずつ別の問いに答える形で図を見ていきます。

- 学習が進むにつれて損失はどう下がるか
- 勉強時間と点数には関係がありそうか
- データはどんな分布をしているか
- クラスごとのばらつきは違うか
- 特徴量どうしは似た動きをしているか

こうして問いを先に置くと、グラフの種類が『選ぶもの』になって、暗記項目ではなくなります。


## 図の部品を最初に押さえる

`fig` は図全体、`ax` は実際に描画する座標領域です。`plt.subplots()` を使うのは、この 2 つを最初に受け取って、『どの図全体に、どの軸へ描くのか』を明示しやすいからです。

また、ヒストグラムの `bins` は分布をどれだけ細かく区切るか、`hue` は同じ図の中でカテゴリを色分けして比較するかを表します。


## 眺めるポイントを決めてから図を見る

可視化のセルでは、コードそのものより『この図から何を読み取るべきか』を先に意識すると理解しやすくなります。タイトルや軸ラベルは、その図が何を比較しているのかを固定するために付いています。


## 同じデータでも、図を変えると見えるものが変わる

このあと出てくる `alpha`, `marker`, `bins`, `hue`, `annot` などの引数は、飾りではなく読解を助けるための調整です。点を透かせば重なりが見えやすくなり、bin 数を変えれば分布の粗さが変わり、heatmap に数値を書き込めば色の印象だけで判断せずに済みます。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")


最初は折れ線グラフです。時間やエポックに沿った変化を見るときの基本形で、訓練損失や精度の推移を眺める場面にそのままつながります。

In [ ]:
days = np.arange(1, 11)
loss = np.array([1.8, 1.5, 1.3, 1.15, 1.05, 0.96, 0.90, 0.86, 0.82, 0.79])

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(days, loss, marker="o", linewidth=2)
ax.set_title("Training Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
plt.show()


散布図は、2 つの量の関係を疑うときの第一選択です。完全な直線になる必要はなく、右上がり・右下がり・まとまりの有無といった『傾向』を読むのが役目です。

In [ ]:
rng = np.random.default_rng(7)
hours = rng.uniform(1, 8, size=80)
score = 52 + hours * 6 + rng.normal(0, 6, size=80)

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.scatter(hours, score, alpha=0.75)
ax.set_title("Study Hours vs Score")
ax.set_xlabel("Hours")
ax.set_ylabel("Score")
plt.show()


ヒストグラムは、平均値だけでは見えない分布の形を見るための図です。裾が長いのか、山が 1 つなのか、ばらつきが大きいのか。`bins` を変えると印象も変わるので、1 枚だけで決めつけない姿勢が大事です。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
axes[0].hist(score, bins=8, color="#4C78A8", alpha=0.85)
axes[0].set_title("bins=8")
axes[1].hist(score, bins=20, color="#F58518", alpha=0.85)
axes[1].set_title("bins=20")
for ax in axes:
    ax.set_xlabel("Score")
axes[0].set_ylabel("Count")
plt.tight_layout()
plt.show()


カテゴリ比較では、各グループの中心とばらつきを同時に見たいことが多くなります。箱ひげ図はそのための図で、中央値と広がりを短時間で比べられます。Seaborn を使うと、この手の図をかなり短いコードで書けます。


In [ ]:
df = pd.DataFrame({
    "class": np.repeat(["A", "B", "C"], repeats=30),
    "score": np.concatenate([
        rng.normal(70, 7, size=30),
        rng.normal(78, 6, size=30),
        rng.normal(74, 8, size=30),
    ])
})

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x="class", y="score", ax=ax)
ax.set_title("Score Distribution by Class")
plt.show()


複数系列を 1 枚に重ねるときは、`hue` や凡例で『どれがどれか』を迷わせないことが重要です。ここでは baseline と improved を同じ軸へ置き、改善後の曲線がどこで差を作っているかを見ます。

In [ ]:
df2 = pd.DataFrame({
    "week": np.tile(np.arange(1, 7), 2),
    "accuracy": np.concatenate([
        np.array([0.61, 0.66, 0.70, 0.74, 0.77, 0.79]),
        np.array([0.58, 0.64, 0.69, 0.72, 0.75, 0.78]),
    ]),
    "model": ["baseline"] * 6 + ["improved"] * 6,
})

fig, ax = plt.subplots(figsize=(6, 3.8))
sns.lineplot(data=df2, x="week", y="accuracy", hue="model", marker="o", ax=ax)
ax.set_ylim(0.55, 0.82)
ax.set_title("Validation Accuracy")
plt.show()


相関行列のヒートマップは、特徴量どうしの似た動きを俯瞰するための図です。`annot=True` を付けると数値も直接読めるので、色の濃さだけで誤解しにくくなります。

In [ ]:
feature_df = pd.DataFrame({
    "math": rng.normal(75, 10, size=120),
    "english": rng.normal(73, 9, size=120),
    "science": rng.normal(70, 11, size=120),
    "study_hours": rng.normal(4.5, 1.2, size=120),
})

corr = feature_df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(5, 4.2))
sns.heatmap(corr, annot=True, cmap="Blues", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation Matrix")
plt.show()


可視化で一番避けたいのは、『描いたけれど何を見る図か自分で言えない』状態です。図の種類を覚えるより、増減・関係・分布・比較・相関のどれを見たいのかを先に言葉にする方が、実務ではずっと役に立ちます。